# Dask Delayed

In [2]:
from pathlib import Path
import time
import xml.etree.ElementTree as ET
from datetime import datetime

import pandas as pd

# Пытаемся импортировать dask, но ноутбук корректно живёт и без него (для проверки логики)
try:
    from dask import delayed
    import dask.bag as db
    import dask.dataframe as dd
    DASK_AVAILABLE = True
except ImportError:
    delayed = None
    db = None
    dd = None
    DASK_AVAILABLE = False
    print("⚠️ Библиотека dask не установлена. Код с dask будет работать после её установки (pip install dask).")

# Пути к файлам (предполагаем, что ноутбук лежит в одной папке с данными)
data_dir = Path(".")
xml_paths = sorted(data_dir.glob("reviewers_full_*.xml"))
review_paths = sorted(data_dir.glob("reviews_*.json"))

print("XML-файлы с пользователями:")
for p in xml_paths:
    print(" •", p)

print("\nJSON-файлы с отзывами:")
for p in review_paths:
    print(" •", p)

XML-файлы с пользователями:
 • reviewers_full_0.xml
 • reviewers_full_1.xml
 • reviewers_full_2.xml
 • reviewers_full_3.xml
 • reviewers_full_4.xml

JSON-файлы с отзывами:
 • reviews_0.json
 • reviews_1.json
 • reviews_2.json


## Лабораторная работа 14

1. Напишите функцию, которая считывает файл формата xml из архива `reviewers_full.zip` и по данным этого файла формирует список словарей, содержащих следующие ключи: `username`, `name`, `sex`, `country`, `mail`, `registered`, `birthdate`, `name_prefix`, `country_code`. Часть из этих значений в исходном файле хранится в виде тэгов, часть - в виде атрибутов тэгов. Для конкретного человека какие-то из этих ключей могут отсутствовать. 



In [3]:
def parse_reviewers_xml(path):
    '''
    Читает XML-файл с пользователями и возвращает список словарей
    с ключами:
    id, username, name, sex, country, mail, registered,
    birthdate, name_prefix, country_code.
    '''
    tree = ET.parse(path)
    root = tree.getroot()
    
    users = []
    for user in root.findall("user"):
        rec = {
            "id": None,
            "username": None,
            "name": None,
            "sex": None,
            "country": None,
            "mail": None,
            "registered": None,
            "birthdate": None,
            "name_prefix": None,
            "country_code": None,
        }
        
        # Префикс имени хранится в атрибуте тега <user>
        prefix = user.attrib.get("prefix")
        if prefix:
            rec["name_prefix"] = prefix
        
        # Остальные поля — в дочерних тегах
        for child in user:
            tag = child.tag
            text = (child.text or "").strip() or None
            
            if tag == "country":
                rec["country"] = text
                rec["country_code"] = child.attrib.get("code")
            elif tag in ("username", "name", "sex", "mail", "registered", "birthdate", "id"):
                rec[tag] = text
        
        users.append(rec)
    
    return users

# Быстрая проверка на одном файле (если он есть)
if xml_paths:
    test_users = parse_reviewers_xml(xml_paths[0])
    print(f"Пример: всего записей в {xml_paths[0].name}: {len(test_users)}")
    print("Первые 3 записи:")
    for u in test_users[:3]:
        print(u)
else:
    print("XML-файлы не найдены (проверьте расположение данных).")

Пример: всего записей в reviewers_full_0.xml: 45314
Первые 3 записи:
{'id': '556011', 'username': 'gabrielacalhoun', 'name': None, 'sex': 'F', 'country': None, 'mail': None, 'registered': None, 'birthdate': '1988-01-25', 'name_prefix': 'Mrs.', 'country_code': None}
{'id': '1251087', 'username': 'qbaxter', 'name': None, 'sex': None, 'country': 'Norway', 'mail': 'qware@gmail.com', 'registered': None, 'birthdate': '1985-01-19', 'name_prefix': None, 'country_code': 'NO'}
{'id': '537188', 'username': 'crosschristopher', 'name': 'Dana Moore', 'sex': None, 'country': None, 'mail': 'stephaniestrong@yahoo.com', 'registered': '2018-11-21', 'birthdate': '1955-07-03', 'name_prefix': None, 'country_code': None}


2. Измерьте время выполнения функции из задания 1 на всех файлах из архива. Ускорьте время выполнения, используя `dask.delayed`.

In [4]:
def parse_all_sequential(paths):
    all_users = []
    t0 = time.perf_counter()
    for p in paths:
        users = parse_reviewers_xml(p)
        all_users.extend(users)
    t1 = time.perf_counter()
    print(f"Последовательная обработка {len(paths)} файлов заняла {t1 - t0:.3f} c")
    print(f"Всего пользователей: {len(all_users)}")
    return all_users

all_users_seq = parse_all_sequential(xml_paths)

# Та же логика, но с использованием dask.delayed
if DASK_AVAILABLE and xml_paths:
    @delayed
    def parse_reviewers_xml_delayed(path):
        return parse_reviewers_xml(path)
    
    reviewers_delayed = [parse_reviewers_xml_delayed(p) for p in xml_paths]
    reviewers_bag_for_timing = db.from_delayed(reviewers_delayed)
    
    t0 = time.perf_counter()
    # bag.compute() вернёт список словарей (flatten по всем партициям)
    all_users_dask = list(reviewers_bag_for_timing)
    t1 = time.perf_counter()
    
    print(f"\nОбработка с dask.delayed заняла {t1 - t0:.3f} c")
    print(f"Всего пользователей (через dask.bag): {len(all_users_dask)}")
else:
    print("\nDask недоступен — пропускаем измерение времени с dask.delayed.")

Последовательная обработка 5 файлов заняла 1.725 c
Всего пользователей: 226570

Обработка с dask.delayed заняла 3.767 c
Всего пользователей (через dask.bag): 226570


3. Задекорируйте функцию из задания 1 при помощи `dask.delayed` и создайте список `reviewers`, состоящий из 5 объектов `delayed` (по одному объекту на файл). Из списка объектов `delayed`, создайте `dask.bag` при помощи метода `db.from_delayed`. Добавьте ключ `birth_year`, в котором хранится год рождения человека. Оставьте в выборке только тех людей, которые __наверняка__ моложе 1980 года. Преобразуйте поле `id` к целому типу.

In [5]:
def add_birth_year(user):
    '''Добавляет поле birth_year (если возможно распарсить дату).'''
    user = dict(user)  # на всякий случай делаем копию
    bd = user.get("birthdate")
    year = None
    if bd:
        try:
            year = int(bd[:4])
        except ValueError:
            year = None
    user["birth_year"] = year
    return user

def born_after_1980(user):
    '''Фильтр: берём только тех, у кого год рождения известен и больше 1980.'''
    year = user.get("birth_year")
    return year is not None and year > 1980

def id_to_int(user):
    '''Преобразует поле id к int (если возможно).'''
    user = dict(user)
    if user.get("id") is not None:
        try:
            user["id"] = int(user["id"])
        except ValueError:
            user["id"] = None
    return user

# Вариант через обычные списки (для проверки логики без dask)
filtered_users_seq = []
for u in all_users_seq:
    u2 = add_birth_year(u)
    if born_after_1980(u2):
        u3 = id_to_int(u2)
        filtered_users_seq.append(u3)

print(f"Пользователей, родившихся после 1980 года (по списку): {len(filtered_users_seq)}")

# Вариант через dask.bag (основное решение)
if DASK_AVAILABLE and xml_paths:
    @delayed
    def parse_reviewers_xml_delayed(path):
        return parse_reviewers_xml(path)
    
    reviewers = [parse_reviewers_xml_delayed(p) for p in xml_paths]
    reviewers_bag = db.from_delayed(reviewers)
    
    reviewers_bag = (
        reviewers_bag
        .map(add_birth_year)     # добавляем birth_year
        .filter(born_after_1980) # оставляем только рождённых после 1980 года
        .map(id_to_int)          # приводим id к int
    )
    
    # Для контроля можно посчитать количество элементов
    n_reviewers = reviewers_bag.count().compute()
    print(f"Пользователей, родившихся после 1980 года (через dask.bag): {n_reviewers}")
else:
    reviewers_bag = None
    print("Dask недоступен — dask.bag не создаётся (логика отработана через обычный список).")

Пользователей, родившихся после 1980 года (по списку): 64088
Пользователей, родившихся после 1980 года (через dask.bag): 64088


4. Из `dask.bag`, полученного в задании 3, создайте `dask.dataframe` при помощи метода `bag.to_dataframe`. Укажите столбец `id` в качестве индекса.

In [7]:
# Вариант через pandas (чтобы сразу получить таблицу и посмотреть на неё)
reviewers_df = pd.DataFrame(filtered_users_seq)
if not reviewers_df.empty:
    reviewers_df = reviewers_df.set_index("id").sort_index()
    print("Таблица reviewers_df (pandas) — первые строки:")
    display(reviewers_df.head())
else:
    print("reviewers_df пустой — возможно, нет пользователей с годом рождения > 1980.")

# Вариант через dask (основное решение)
if DASK_AVAILABLE and reviewers_bag is not None:
    reviewers_dd = reviewers_bag.to_dataframe()
    # индекс по полю id
    reviewers_dd = reviewers_dd.set_index("id")
    print("\nDask DataFrame (reviewers_dd) успешно создан. Его первая часть (через .compute().head()):")
    display(reviewers_dd.compute().head())
else:
    print("\nDask недоступен — dask.dataframe не создаётся, работаем с pandas DataFrame reviewers_df.")

Таблица reviewers_df (pandas) — первые строки:


,username,name,sex,country,mail,registered,birthdate,name_prefix,country_code,birth_year
id,,,,,,,,,,
1676,lgeorge,None,M,None,None,None,1983-06-24,None,None,1983
1792,qbeard,None,F,Guinea,rachel20@hotmail.com,None,1986-03-12,None,GN,1986
1938,adambrown,William Fisher,None,New Caledonia,None,2019-05-03,1991-11-11,None,NC,1991
2046,vthompson,Emily Sanford,F,United Arab Emirates,omelendez@yahoo.com,2001-10-30,1981-11-27,None,AE,1981
2095,djohnson,Jennifer Hawkins,F,Jamaica,None,None,1984-09-23,Mrs.,JM,1984



Dask DataFrame (reviewers_dd) успешно создан. Его первая часть (через .compute().head()):


,username,name,sex,country,mail,registered,birthdate,name_prefix,country_code,birth_year
id,,,,,,,,,,
1676,lgeorge,None,M,None,None,None,1983-06-24,None,None,1983
1792,qbeard,None,F,Guinea,rachel20@hotmail.com,None,1986-03-12,None,GN,1986
1938,adambrown,William Fisher,None,New Caledonia,None,2019-05-03,1991-11-11,None,NC,1991
2046,vthompson,Emily Sanford,F,United Arab Emirates,omelendez@yahoo.com,2001-10-30,1981-11-27,None,AE,1981
2095,djohnson,Jennifer Hawkins,F,Jamaica,None,None,1984-09-23,Mrs.,JM,1984


5. Назовем отзыв негативным, если оценка равна 0, 1 или 2. Загрузите данные о негативных отзывах из файлов архива `reviews_full` (__ЛР12__) в виде `dask.DataFrame`. Посчитайте количество отзывов с группировкой по пользователю, оставившему отзыв. Объедините результат с таблицей, полученной в задаче 4.

In [10]:
# --- Вариант через pandas (для немедленного получения результата) ---
review_dfs = []
for p in review_paths:
    if p.exists():
        df = pd.read_json(p, lines=True)
        review_dfs.append(df)
    else:
        print(f"Файл {p} не найден.")

if review_dfs:
    reviews_df = pd.concat(review_dfs, ignore_index=True)
    print(f"Всего строк в reviews_df: {len(reviews_df)}")
    display(reviews_df.head())
else:
    reviews_df = pd.DataFrame()
    print("Файлы с отзывами не найдены.")

# Предполагаем, что все эти отзывы уже негативные (по условию задачи).
# Считаем количество отзывов по каждому пользователю:
if not reviews_df.empty:
    neg_counts = (
        reviews_df
        .groupby("user_id")
        .size()
        .rename("n_negative_reviews")
        .to_frame()
    )
    print("\nЧисло негативных отзывов по каждому user_id (pandas):")
    display(neg_counts.head())
else:
    neg_counts = pd.DataFrame(columns=["user_id", "n_negative_reviews"])
    print("reviews_df пустой — не из чего считать статистику.")

# Объединяем с таблицей пользователей (reviewers_df), где индексом является id пользователя
if not reviewers_df.empty and not neg_counts.empty:
    result_df = reviewers_df.join(neg_counts, how="left")
    print("\nИтоговая таблица пользователей с числом негативных отзывов:")
    display(result_df.head())
else:
    result_df = reviewers_df.copy()
    if "n_negative_reviews" not in result_df.columns:
        result_df["n_negative_reviews"] = pd.NA
    print("\nНе удалось объединить данные (одна из таблиц пуста).")

# --- Вариант через dask (основное решение) ---
if DASK_AVAILABLE and review_paths and xml_paths:
    # Загружаем негативные отзывы как dask.DataFrame
    reviews_dd = dd.read_json(str(data_dir / "reviews_*.json"), lines=True)
    
    # Считаем количество негативных отзывов по каждому user_id
    neg_counts_dd = (
        reviews_dd
        .groupby("user_id")
        .size()
        .rename("n_negative_reviews")
        .to_frame()
    )
    
    # Объединяем с reviewers_dd (dask.DataFrame из задачи 4)
    if 'reviewers_dd' in globals():
        joined_dd = reviewers_dd.join(neg_counts_dd, how="left")
        print("\nПример результата объединения (через dask, .compute().head()):")
        display(joined_dd.compute().head())
    else:
        print("\nreviewers_dd не создан (dask.bag недоступен ранее). Можно объединить вручную через pandas.")
else:
    print("\nDask недоступен — объединение показывается только в pandas-варианте.")

Всего строк в reviews_df: 702627


,user_id,recipe_id,date,review
0,452355,292657,2016-05-08,WOW!!! This is the best. I have never been abl...
1,329304,433404,2006-06-14,This was good but the dressing needed somethin...
2,227932,2008187,1985-11-19,"Very good,it was a hit for my family. I used 6..."
3,171468,270716,2019-05-21,Made for ZWT-8 Family Picks after I saw these ...
4,91392,1159916,1972-09-18,Very nice slaw. I especially like that it does...



Число негативных отзывов по каждому user_id (pandas):


,n_negative_reviews
user_id,
1533,64
1535,441
1581,1
1634,36
1676,29



Итоговая таблица пользователей с числом негативных отзывов:


,username,name,sex,country,mail,registered,birthdate,name_prefix,country_code,birth_year,n_negative_reviews
id,,,,,,,,,,,
1676,lgeorge,None,M,None,None,None,1983-06-24,None,None,1983,29.0
1792,qbeard,None,F,Guinea,rachel20@hotmail.com,None,1986-03-12,None,GN,1986,14.0
1938,adambrown,William Fisher,None,New Caledonia,None,2019-05-03,1991-11-11,None,NC,1991,3.0
2046,vthompson,Emily Sanford,F,United Arab Emirates,omelendez@yahoo.com,2001-10-30,1981-11-27,None,AE,1981,3.0
2095,djohnson,Jennifer Hawkins,F,Jamaica,None,None,1984-09-23,Mrs.,JM,1984,NaN



Пример результата объединения (через dask, .compute().head()):


,username,name,sex,country,mail,registered,birthdate,name_prefix,country_code,birth_year,n_negative_reviews
id,,,,,,,,,,,
1676,lgeorge,None,M,None,None,None,1983-06-24,None,None,1983,29.0
1792,qbeard,None,F,Guinea,rachel20@hotmail.com,None,1986-03-12,None,GN,1986,14.0
1938,adambrown,William Fisher,None,New Caledonia,None,2019-05-03,1991-11-11,None,NC,1991,3.0
2046,vthompson,Emily Sanford,F,United Arab Emirates,omelendez@yahoo.com,2001-10-30,1981-11-27,None,AE,1981,3.0
2095,djohnson,Jennifer Hawkins,F,Jamaica,None,None,1984-09-23,Mrs.,JM,1984,NaN
